## Reading Data from Azure Cosmos DB using PySpark

In [ ]:
endpoint="https://supplychaincosmosdb.documents.azure.com:443/"
key="d0giVUTnimAtnRR3fwtCuQJwPhHUcRExRq46j3W9UCvIFzIKDxo8OdAubCZ3iyJAdJ5p3UhNR8WbACDbKDigFw=="
database="supplychain"
container1="supplychaindata"
container2="supplychainlogs"
dataConfig={
  "spark.cosmos.accountEndpoint": endpoint,
  "spark.cosmos.accountKey": key,
  "spark.cosmos.database": database,
  "spark.cosmos.container": container1
}
logsConfig={
  "spark.cosmos.accountEndpoint": endpoint,
  "spark.cosmos.accountKey": key,
  "spark.cosmos.database": database,
  "spark.cosmos.container": container2
}
df_data = spark.read.format("cosmos.oltp").options(**dataConfig).load()
df_logs = spark.read.format("cosmos.oltp").options(**logsConfig).load()

## Data Cleaning

In [ ]:
from pyspark.sql.functions import col,when,sum,to_timestamp,desc,trim,count,avg,coalesce,month,year

In [ ]:
column_null_counts = df_data.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_data.columns
])

column_null_dict = column_null_counts.first().asDict()
for c, v in column_null_dict.items():
    if v>0:
        print(c,v)

Customer Zipcode 3
Product Description 180519
Customer Lname 8
Order Zipcode 155679


In [ ]:
drop_columns = [
    "Customer Password",
    "Customer Email",
    "Product Image",
    "Product Description",
    "Order Zipcode"
]

df_clean = df_data.drop(*drop_columns)
df_clean=df_clean.dropna()

In [ ]:
double_fields = [
    "Product Price",
    "Order Item Product Price",
    "Sales",
    "Latitude",
    "Longitude",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Profit Ratio",
    "Benefit per order",
    "Order Profit Per Order",
    "Order Item Total",
    "Sales per customer"
]
integer_fields = [
    "Order Item Quantity",
    "Days for shipment (scheduled)",
    "Days for shipping (real)"
]
date_fields = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]
df_casted = df_clean
for col_name in double_fields:
    df_casted = df_casted.withColumn(col_name, col(col_name).cast("double"))
for col_name in integer_fields:
    df_casted = df_casted.withColumn(col_name, col(col_name).cast("int"))
for col_name in date_fields:
    df_casted = df_casted.withColumn(
        col_name,
        coalesce(
            to_timestamp(col(col_name), "dd-MM-yyyy HH:mm"),
            to_timestamp(col(col_name), "M/d/yyyy H:mm")
        )
    )
    
string_columns = [c for c, t in df_casted.dtypes if t == "string"]

for c in string_columns:
    df_casted = df_casted.withColumn(c, trim(col(c)))

In [ ]:
customer_cols = [
    "Customer Id", "Customer Fname", "Customer Lname",
    "Customer Segment", "Customer Country", "Customer State", 
    "Customer City", "Customer Street", "Customer Zipcode"
]
customer_df = df_casted.select(customer_cols)

order_cols = [
    "Order Id", "Customer Id", "Order Country", "Order State", "Order City",
    "Order Region", "Order Status", "Delivery Status", "Late_delivery_risk",
    "order date (DateOrders)", "shipping date (DateOrders)", "Shipping Mode",
    "Days for shipment (scheduled)", "Days for shipping (real)", 
    "Benefit per order", "Order Profit Per Order","Sales", "Sales per customer",
]
order_df = df_casted.select(order_cols)

product_cols = [
    "Product Card Id", "Product Name", "Product Category Id",
    "Category Name", "Department Id", "Department Name",
    "Type", "Product Status","Product Price"
]

product_df = df_casted.select(product_cols)

order_item_cols = [
    "Order Item Id", "Order Id", "Product Card Id", "Order Item Product Price",
    "Order Item Quantity", "Order Item Discount", "Order Item Discount Rate",
    "Order Item Profit Ratio", "Order Item Total", "Market"
]
order_item_df = df_casted.select(order_item_cols)

In [ ]:
logs_df=df_logs.withColumn(
    "date_parsed",
    coalesce(
        to_timestamp(col("Date"), "dd-MM-yyyy HH:mm:ss"),
        to_timestamp(col("Date"), "M/d/yyyy H:m")
    )
)

## Customer Segment Distribution Analysis

In [ ]:
cust_seg=customer_df.groupBy("Customer Segment").count()
display(cust_seg)

Customer Segment,count
Consumer,93501
Home Office,32223
Corporate,54784


Databricks visualization. Run in Databricks to view.

## Top Customer Locations

In [ ]:
cust_country = customer_df.groupBy("Customer Country","Customer City").count()\
    .orderBy(desc("count")).limit(15)
display(cust_country)

Customer Country,Customer City,count
Puerto Rico,Caguas,66768
EE. UU.,Chicago,3885
EE. UU.,Los Angeles,3416
EE. UU.,Brooklyn,3412
EE. UU.,New York,1816
EE. UU.,Philadelphia,1577
EE. UU.,Bronx,1500
EE. UU.,San Diego,1437
EE. UU.,Miami,1314
EE. UU.,Houston,1297


Databricks visualization. Run in Databricks to view.

## Top Products by Total Sales Revenue

In [ ]:

product_sales = order_item_df.join(product_df, "Product Card Id") \
    .groupBy("Product Card Id", "Product Name") \
    .agg(
        sum("Order Item Total").alias("total_sales"),
        sum("Order Item Quantity").alias("units_sold"),
        avg("Order Item Profit Ratio").alias("avg_profit_ratio")
    ) 

In [ ]:
sales_perf=product_sales.orderBy(desc("total_sales")).limit(15)

display(sales_perf)

Product Card Id,Product Name,total_sales,units_sold,avg_profit_ratio
1004,Field & Stream Sportsman 16 Gun Fire Safe,1.0788165390909616E11,300155625,0.12144935061084695
365,Perfect Fitness Perfect Rip Deck,9.740251659336526E10,1806706470,0.12466041193093924
502,Nike Men's Dri-FIT Victory Golf Polo,5.95018832975E10,1324279460,0.12219348704271625
403,Nike Men's CJ Elite 2 TD Football Cleat,5.780610688302276E10,494884516,0.12012316819183122
957,Diamondback Women's Serene Classic Comfort Bi,5.080805843510061E10,188485441,0.11594726500595227
1014,O'Brien Men's Neoprene Life Vest,5.010636960339892E10,1115482294,0.12393615910746673
1073,Pelican Sunstream 100 Kayak,4.317553032138585E10,240250000,0.11657419353266434
191,Nike Men's Free 5.0+ Running Shoe,4.0105291751490395E10,446358920,0.11898183900803601
627,Under Armour Girls' Toddler Spine Surge Runni,1.2111565248247446E10,336930495,0.11250070631261107
1351,Dell Laptop,2.6316459E8,195364,0.11705882384615293


Databricks visualization. Run in Databricks to view.

## Top Products by Uints Sold

In [ ]:
units_perf=product_sales.orderBy(desc("units_sold")).limit(15)

display(units_perf)

Product Card Id,Product Name,total_sales,units_sold,avg_profit_ratio
365,Perfect Fitness Perfect Rip Deck,9.740251659336526E10,1806706470,0.12466041193093924
502,Nike Men's Dri-FIT Victory Golf Polo,5.95018832975E10,1324279460,0.12219348704271625
1014,O'Brien Men's Neoprene Life Vest,5.010636960339892E10,1115482294,0.12393615910746673
403,Nike Men's CJ Elite 2 TD Football Cleat,5.780610688302276E10,494884516,0.12012316819183122
191,Nike Men's Free 5.0+ Running Shoe,4.0105291751490395E10,446358920,0.11898183900803601
627,Under Armour Girls' Toddler Spine Surge Runni,1.2111565248247446E10,336930495,0.11250070631261107
1004,Field & Stream Sportsman 16 Gun Fire Safe,1.0788165390909616E11,300155625,0.12144935061084695
1073,Pelican Sunstream 100 Kayak,4.317553032138585E10,240250000,0.11657419353266434
957,Diamondback Women's Serene Classic Comfort Bi,5.080805843510061E10,188485441,0.11594726500595227
1362,Fighting video games,2.5009760770980783E7,700569,0.09146953391756413


Databricks visualization. Run in Databricks to view.

## Shipment Delay Analysis

In [ ]:
delay_df = order_df
delay_df = delay_df.withColumn(
    "Shipment_delay_days",
    col("Days for shipping (real)") - col("Days for shipment (scheduled)")
).withColumn(
    "Shipment_delay_status",
    when(col("Shipment_delay_days") > 0, "Late")
    .when(col("Shipment_delay_days") == 0, "On Time")
    .otherwise("Early")
).select( "Order Id", "Customer Id","Order City",
    "Order Region", "Shipping Mode","Shipment_delay_status","Shipment_delay_days")

display(delay_df)

Order Id,Customer Id,Order City,Order Region,Shipping Mode,Shipment_delay_status,Shipment_delay_days
63295,11731,Berl�n,Western Europe,First Class,Late,1
31959,5268,Arlington,US Center,Standard Class,Late,1
75745,19298,Jiangyan,Eastern Asia,Standard Class,Late,2
46073,10834,Sincan,West Asia,Standard Class,On Time,0
12540,6758,Peterborough,Northern Europe,Standard Class,Late,2
24548,6763,Daegu,Eastern Asia,Standard Class,On Time,0
55665,7161,Managua,Central America,Standard Class,Late,2
10794,3410,Tours,Western Europe,Standard Class,Late,1
50623,12231,Ufa,Eastern Europe,Second Class,Late,2
29474,11354,Nasik,South Asia,Second Class,Late,3


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## Late Delivery Risk Distribution

In [ ]:
risk_df=order_df
delivery_risk = risk_df.groupBy("Late_delivery_risk") \
    .agg(count("*").alias("total_orders")) \
    .withColumn(
        "Late_delivery_risk",
        when(col("Late_delivery_risk") == 0, "No Risk")
        .when(col("Late_delivery_risk") == 1, "At Risk")
        )
display(delivery_risk)

Late_delivery_risk,total_orders
No Risk,81536
At Risk,98972


Databricks visualization. Run in Databricks to view.

## On-Time Delivery Performance by Location

In [ ]:
order_df = order_df.withColumn(
    "on_time",
    (col("Delivery Status") == "Shipping on time") | (col("Late_delivery_risk") == 0)
)

In [ ]:
delivery_performance = order_df.groupBy("Order Country", "Order City") \
    .agg(
        count("*").alias("total_orders"),
        sum(col("on_time").cast("integer")).alias("on_time_orders")
    ) \
    .withColumn("on_time_percent", (col("on_time_orders") / col("total_orders") * 100)) \
    .orderBy(desc("on_time_percent"))
display(delivery_performance)

Order Country,Order City,total_orders,on_time_orders,on_time_percent
Brasil,Itamaraju,7,7,100.0
China,Loudi,3,3,100.0
China,Huadian,4,4,100.0
Alemania,Willich,3,3,100.0
Colombia,Neiva,13,13,100.0
Francia,Pierrefitte-sur-Seine,17,17,100.0
Turqu�a,Ardahan,5,5,100.0
Albania,Shkoder,10,10,100.0
Estados Unidos,Saint Peters,1,1,100.0
Francia,Voiron,2,2,100.0


Databricks visualization. Run in Databricks to view.

## On-Time Delivery Performance by Shipping Mode

In [ ]:
shipping_performance = order_df.groupBy("Shipping Mode") \
    .agg(
        count("*").alias("total_orders"),
        sum(col("on_time").cast("integer")).alias("on_time_orders")
    ) \
    .withColumn("on_time_percent", (col("on_time_orders") / col("total_orders") * 100)) \
    .orderBy(desc("on_time_percent"))
display(shipping_performance)

Shipping Mode,total_orders,on_time_orders,on_time_percent
Standard Class,107745,66724,61.92769966123718
Same Day,9737,5283,54.256957995275755
Second Class,35214,8228,23.365706821150678
First Class,27812,1301,4.677836904933122


Databricks visualization. Run in Databricks to view.

## Most Viewed Products Based on User Logs

In [ ]:
top_products = logs_df.groupBy("Product") \
    .count() \
    .orderBy(desc("count"))

display(top_products)

Product,count
Perfect Fitness Perfect Rip Deck,27878
adidas Kids' RG III Mid Football Cleat,26200
Nike Men's Dri-FIT Victory Golf Polo,25627
Nike Men's CJ Elite 2 TD Football Cleat,25241
O'Brien Men's Neoprene Life Vest,16194
Pelican Sunstream 100 Kayak,16186
Diamondback Women's Serene Classic Comfort Bi,15521
Field & Stream Sportsman 16 Gun Fire Safe,15178
Under Armour Hustle Storm Medium Duffle Bag,13752
Columbia Men's PFG Anchor Tough T-Shirt,13716


Databricks visualization. Run in Databricks to view.

## Monthly User Activity Trend

In [ ]:
monthly_trend = logs_df.groupBy("Month") \
    .count() \
    .orderBy("Month")

display(monthly_trend)

Month,count
Dec,84093
Jan,83581
Nov,80860
Oct,84205
Sep,137238


Databricks visualization. Run in Databricks to view.

## Peak Traffic Hours

In [ ]:
peak_hours = logs_df.groupBy("Hour")\
    .count()\
    .orderBy(desc("count"))

display(peak_hours)

Hour,count
21,30749
20,30551
22,29682
19,28292
23,27778
18,25575
11,24801
17,24544
12,24422
10,24036


Databricks visualization. Run in Databricks to view.

## Analysis of Viewed vs Added-to-Cart Behavior by Category for period Jan-2018

In [ ]:

df_jan2018 = logs_df.filter((month(col("date_parsed")) == 1) & (year(col("date_parsed")) == 2018))
df_jan2018 = df_jan2018.withColumn(
    "add_to_cart_flag",
    when(col("url").contains("/add_to_cart"), 1).otherwise(0)
)

In [ ]:
category_summary = df_jan2018.groupBy("Category") \
    .agg(
        count(when(col("add_to_cart_flag") == 0, True)).alias("Viewed_count"),
        sum("add_to_cart_flag").alias("Add_to_cart_count")
    ) \
    .withColumn(
        "conversion_rate",
        col("Add_to_cart_count") / col("Viewed_count")
    )\
    .orderBy(col("conversion_rate"))

display(category_summary)

Category,Viewed_count,Add_to_cart_count,conversion_rate
baseball & softball,1616,574,0.3551980198019802
as seen on tv!,1720,635,0.3691860465116279
cardio equipment,1777,665,0.37422622397298816
camping & hiking,1988,744,0.37424547283702214
cleats,3584,1342,0.3744419642857143
strength training,1608,604,0.3756218905472637
boxing & mma,1588,601,0.378463476070529
basketball,1591,603,0.37900691389063484
lacrosse,1627,619,0.38045482483097726
fitness accessories,1751,676,0.38606510565391206


Databricks visualization. Run in Databricks to view.

## Writing DataFrames to Azure SQL Database

In [ ]:
jdbc_hostname = "dataserver.database.windows.net"
jdbc_port = 1433                 
database_name = "supplydb"
username = "jeeva"
password = "admin@123"

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};databaseName={database_name}"

connection_properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver" 
}

In [ ]:
customer_df.write.jdbc(
    url=jdbc_url,
    table="customer_table",
    mode="overwrite",  
    properties=connection_properties
)
order_df.write.jdbc(
    url=jdbc_url,
    table="order_table",
    mode="overwrite", 
    properties=connection_properties
)
product_df.write.jdbc(
    url=jdbc_url,
    table="product_table",
    mode="overwrite", 
    properties=connection_properties
)
order_item_df.write.jdbc(
    url=jdbc_url,
    table="order_item_table",
    mode="overwrite", 
    properties=connection_properties
)
logs_df.write.jdbc(
    url=jdbc_url,
    table="logs_table",
    mode="overwrite", 
    properties=connection_properties
)

## Reading DataFrames to Azure SQL Database

In [ ]:
df=spark.read.jdbc(
    url=jdbc_url,
    table="logs_table",
    properties=connection_properties
)

display(df)

Department,ip,Category,url,Month,id,Date,Product,Hour,date_parsed
apparel,29.38.172.7,featured shops,/department/apparel/category/featured%20shops/product/adidas%20Kids'%20RG%20III%20Mid%20Football%20Cleat,Oct,bd75664e-b57f-4373-8198-55e9946f884b,10/10/2017 11:05,adidas Kids' RG III Mid Football Cleat,11,2017-10-10T11:05:00Z
golf,192.4.253.83,girls' apparel,/department/golf/category/girls'%20apparel/product/adidas%20Youth%20Germany%20Black/Red%20Away%20Match%20Soc,Oct,7b441365-a512-4445-8fec-579949893c8f,10/10/2017 11:04,adidas Youth Germany Black/Red Away Match Soc,11,2017-10-10T11:04:00Z
footwear,160.13.45.93,electronics,/department/footwear/category/electronics/product/Under%20Armour%20Kids'%20Mercenary%20Slide,Oct,c78e4add-e140-4ad7-857a-bcab873ac599,10/10/2017 10:59,Under Armour Kids' Mercenary Slide,10,2017-10-10T10:59:00Z
apparel,102.23.8.25,featured shops,/department/apparel/category/featured%20shops/product/adidas%20Kids'%20RG%20III%20Mid%20Football%20Cleat,Oct,cd222e21-7c26-4628-87f2-2a6666d4f387,10/10/2017 10:59,adidas Kids' RG III Mid Football Cleat,10,2017-10-10T10:59:00Z
golf,28.187.204.212,women's apparel,/department/golf/category/women's%20apparel/product/Nike%20Men's%20Dri-FIT%20Victory%20Golf%20Polo,Oct,9b902487-127d-4823-8321-b940f918aebb,10/10/2017 10:55,Nike Men's Dri-FIT Victory Golf Polo,10,2017-10-10T10:55:00Z
apparel,11.6.19.57,men's footwear,/department/apparel/category/men's%20footwear/product/Nike%20Men's%20CJ%20Elite%202%20TD%20Football%20Cleat/add_to_cart,Oct,403732c4-3aca-4705-b51e-a5c19ade33cb,10/10/2017 10:55,Nike Men's CJ Elite 2 TD Football Cleat,10,2017-10-10T10:55:00Z
footwear,149.167.143.128,boxing & mma,/department/footwear/category/boxing%20&%20mma/product/Under%20Armour%20Women's%20Micro%20G%20Skulpt%20Running%20S,Oct,0cc6a565-296a-454d-98f6-400939fa81cc,10/10/2017 10:51,Under Armour Women's Micro G Skulpt Running S,10,2017-10-10T10:51:00Z
fan shop,25.45.226.91,water sports,/department/fan%20shop/category/water%20sports/product/Pelican%20Sunstream%20100%20Kayak,Oct,76bbb437-b51e-4c7c-82a6-8c6ccb31995f,10/10/2017 10:51,Pelican Sunstream 100 Kayak,10,2017-10-10T10:51:00Z
outdoors,160.13.45.93,golf bags & carts,/department/outdoors/category/golf%20bags%20&%20carts/product/Ogio%20Race%20Golf%20Shoes,Oct,934fbc1d-1406-42aa-96a6-2b9270904884,10/10/2017 10:47,Ogio Race Golf Shoes,10,2017-10-10T10:47:00Z
fan shop,50.73.131.27,camping & hiking,/department/fan%20shop/category/camping%20&%20hiking/product/Diamondback%20Women's%20Serene%20Classic%20Comfort%20Bi,Oct,9113f78a-96a0-4b2f-8406-6fcef240701f,10/10/2017 10:45,Diamondback Women's Serene Classic Comfort Bi,10,2017-10-10T10:45:00Z
